# 🌿 TRAIN TĂNG CƯỜNG DỮ LIỆU CHUYÊN SÂU CHO 48 LOÀI TỶ LỆ THẤP (< 60%) & ÍT ẢNH

**Hướng dẫn chạy trên Google Colab (GPU T4):**
1. Vào menu **Runtime → Change runtime type → T4 GPU**.
2. Nhấn **Runtime → Run all (`Ctrl + F9`)**.
3. Notebook tự động thực hiện quy trình chuẩn chống rò rỉ dữ liệu (*Zero Data Leakage*):
   - **Bước 1:** Kết nối Google Drive, tự động tìm `merged_90_30k_chuan.zip` trong `MyDrive/KiengLa_AI/`.
   - **Bước 2:** Lọc trùng MD5 và chia **Stratified Split 80% Train – 10% Val – 10% Test (`seed = 999`)**. Tập **Val** và **Test** giữ nguyên ảnh gốc 100%.
   - **Bước 3:** **Chỉ trên tập Train**, tự động tăng cường ảnh (**xoay góc ±25°, lật ngang, giảm/tăng độ sáng 0.70x–1.30x, tăng/giảm tương phản, zoom nhẹ**) cho đúng **48 loài ưu tiên** (có chỉ số < 60% hoặc thiếu ảnh).
   - **Bước 4:** Huấn luyện 2 giai đoạn (Feature Extraction → Fine-Tuning) với **Balanced Class Weights** cho `MobileNetV2` (và tùy chọn `EfficientNetB0`).
   - **Bước 5:** Đánh giá trên **tập Test gốc độc lập** và xuất thẳng file Excel kết quả thật `Ket_Qua_Train_Tang_Cuong_48_Loai.xlsx` + model `.keras` vào Google Drive!


In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE, CẤU HÌNH & DANH SÁCH 48 LOÀI ƯU TIÊN
# ==============================================================================
import os, json, math, time, random, shutil, hashlib, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# Cố định Random Seed để tái lập thực nghiệm
RANDOM_SEED = 999
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

# Cấu hình tăng cường dữ liệu CHỈ TRÊN TẬP TRAIN cho 48 loài yếu / ít ảnh
TARGET_TRAIN_PER_PRIORITY_CLASS = 320   # Mỗi loài ưu tiên sẽ được bù tối thiểu 320 ảnh Train
MIN_EXTRA_AUG_FOR_PRIORITY = 80         # Luôn sinh thêm ít nhất 80 biến thể xoay/sáng cho mỗi loài ưu tiên
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Chọn mô hình cần train thật (có thể để ['mobilenet_v2'] cho nhanh, hoặc cả 2)
MODELS_TO_TRAIN = ['mobilenet_v2', 'efficientnet_b0']

# Tự động dò tìm file zip dữ liệu gốc trên Google Drive
PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
candidate_paths = [
    PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'merged_v1' / 'merged_90_30k_chuan.zip',
    PROJECT / '01_Du_Lieu_Goc' / 'merged_90_30k_chuan.zip',
    PROJECT / 'merged_v1' / 'merged_90_30k_chuan.zip',
]
ZIP_30K_PATH = next((p for p in candidate_paths if p.is_file()), None)
if ZIP_30K_PATH is None:
    found = list(PROJECT.rglob('*merged_90_30k*.zip'))
    ZIP_30K_PATH = found[0] if found else None

assert ZIP_30K_PATH is not None and ZIP_30K_PATH.is_file(), f'❌ Không tìm thấy file merged_90_30k*.zip trong {PROJECT}!'

RAW_DIR = Path('/content/raw_30k_extracted')
SPLIT_DIR = Path('/content/data_split_80_10_10_aug48')
SAVE_ROOT = PROJECT / 'train_tang_cuong_48_loai_duoi_60'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Danh sách chính xác 48 loài ưu tiên (< 60% hoặc Train < 180)
PRIORITY_48 = [
  {
    "original_label": "China Doll Plant (Radermachera sinica)",
    "vietnamese_name": "Cây hạnh phúc",
    "baseline_f1": 36.84,
    "baseline_recall": 31.82,
    "train_30k": 169,
    "priority_level": "high"
  },
  {
    "original_label": "Weeping Fig (Ficus benjamina)",
    "vietnamese_name": "Si",
    "baseline_f1": 38.1,
    "baseline_recall": 30.77,
    "train_30k": 199,
    "priority_level": "high"
  },
  {
    "original_label": "Kalanchoe",
    "vietnamese_name": "Cây sống đời",
    "baseline_f1": 40.0,
    "baseline_recall": 31.25,
    "train_30k": 116,
    "priority_level": "high"
  },
  {
    "original_label": "Gold Dust Dracaena (Dracaena surculosa)",
    "vietnamese_name": "Trúc đốm",
    "baseline_f1": 43.9,
    "baseline_recall": 39.13,
    "train_30k": 176,
    "priority_level": "high"
  },
  {
    "original_label": "Fiddle Leaf Fig (Ficus lyrata)",
    "vietnamese_name": "Bàng Singapore",
    "baseline_f1": 46.43,
    "baseline_recall": 50.0,
    "train_30k": 202,
    "priority_level": "high"
  },
  {
    "original_label": "White Bird of Paradise (Strelitzia nicolai)",
    "vietnamese_name": "Thiên điểu trắng",
    "baseline_f1": 46.73,
    "baseline_recall": 89.29,
    "train_30k": 219,
    "priority_level": "high"
  },
  {
    "original_label": "Red-leaf Philodendron (Philodendron erubescens)",
    "vietnamese_name": "Trầu bà đỏ",
    "baseline_f1": 47.22,
    "baseline_recall": 60.71,
    "train_30k": 218,
    "priority_level": "high"
  },
  {
    "original_label": "Silver Pothos (Scindapsus pictus)",
    "vietnamese_name": "Trầu bà bạc",
    "baseline_f1": 48.65,
    "baseline_recall": 39.13,
    "train_30k": 175,
    "priority_level": "high"
  },
  {
    "original_label": "Money Tree (Pachira aquatica)",
    "vietnamese_name": "Kim ngân",
    "baseline_f1": 48.78,
    "baseline_recall": 81.08,
    "train_30k": 285,
    "priority_level": "high"
  },
  {
    "original_label": "Indian Azalea (Rhododendron simsii)",
    "vietnamese_name": "Đỗ quyên",
    "baseline_f1": 48.89,
    "baseline_recall": 40.74,
    "train_30k": 216,
    "priority_level": "high"
  },
  {
    "original_label": "Peppermint (Mentha piperita)",
    "vietnamese_name": "Bạc hà",
    "baseline_f1": 48.89,
    "baseline_recall": 39.29,
    "train_30k": 217,
    "priority_level": "high"
  },
  {
    "original_label": "Chinese Evergreen Siam (Aglaonema commutatum)",
    "vietnamese_name": "Ngân hậu Siam",
    "baseline_f1": 48.98,
    "baseline_recall": 42.86,
    "train_30k": 218,
    "priority_level": "high"
  },
  {
    "original_label": "Swiss Cheese Vine (Monstera adansonii)",
    "vietnamese_name": "Trầu bà lỗ",
    "baseline_f1": 49.12,
    "baseline_recall": 48.28,
    "train_30k": 222,
    "priority_level": "high"
  },
  {
    "original_label": "Song of India (Dracaena reflexa)",
    "vietnamese_name": "Trúc bách hợp",
    "baseline_f1": 50.0,
    "baseline_recall": 48.0,
    "train_30k": 190,
    "priority_level": "medium"
  },
  {
    "original_label": "Staghorn Fern (Platycerium bifurcatum)",
    "vietnamese_name": "Dương xỉ sừng hươu",
    "baseline_f1": 50.0,
    "baseline_recall": 46.15,
    "train_30k": 198,
    "priority_level": "medium"
  },
  {
    "original_label": "Garden Croton (Codiaeum variegatum)",
    "vietnamese_name": "Cô tòng",
    "baseline_f1": 52.0,
    "baseline_recall": 44.83,
    "train_30k": 220,
    "priority_level": "high"
  },
  {
    "original_label": "Ginseng Ficus (Ficus microcarpa)",
    "vietnamese_name": "Sanh bonsai",
    "baseline_f1": 52.83,
    "baseline_recall": 53.85,
    "train_30k": 200,
    "priority_level": "medium"
  },
  {
    "original_label": "Spider Plant (Chlorophytum comosum)",
    "vietnamese_name": "Dây nhện",
    "baseline_f1": 54.9,
    "baseline_recall": 50.0,
    "train_30k": 215,
    "priority_level": "medium"
  },
  {
    "original_label": "Corn Plant (Dracaena fragrans)",
    "vietnamese_name": "Thiết mộc lan",
    "baseline_f1": 55.32,
    "baseline_recall": 50.0,
    "train_30k": 198,
    "priority_level": "medium"
  },
  {
    "original_label": "Rabbit Foot Fern (Davallia fejeensis)",
    "vietnamese_name": "Dương xỉ chân thỏ",
    "baseline_f1": 55.56,
    "baseline_recall": 43.48,
    "train_30k": 180,
    "priority_level": "high"
  },
  {
    "original_label": "Dragon Tail (Epipremnum pinnatum)",
    "vietnamese_name": "Trầu bà đuôi rồng",
    "baseline_f1": 56.14,
    "baseline_recall": 66.67,
    "train_30k": 192,
    "priority_level": "medium"
  },
  {
    "original_label": "Areca Palm (Dypsis lutescens)",
    "vietnamese_name": "Cau vàng",
    "baseline_f1": 56.25,
    "baseline_recall": 45.0,
    "train_30k": 150,
    "priority_level": "medium"
  },
  {
    "original_label": "Zebra Plant (Aphelandra squarrosa)",
    "vietnamese_name": "Cây ngựa vằn Aphelandra",
    "baseline_f1": 56.25,
    "baseline_recall": 40.91,
    "train_30k": 171,
    "priority_level": "high"
  },
  {
    "original_label": "Hairy Philodendron (Philodendron squamiferum)",
    "vietnamese_name": "Trầu bà cuống lông",
    "baseline_f1": 56.41,
    "baseline_recall": 40.74,
    "train_30k": 216,
    "priority_level": "high"
  },
  {
    "original_label": "Schefflera",
    "vietnamese_name": "Ngũ gia bì",
    "baseline_f1": 58.62,
    "baseline_recall": 51.52,
    "train_30k": 260,
    "priority_level": "medium"
  },
  {
    "original_label": "Baby Rubber Plant (Peperomia obtusifolia)",
    "vietnamese_name": "Trường sinh lá tròn",
    "baseline_f1": 59.02,
    "baseline_recall": 64.29,
    "train_30k": 216,
    "priority_level": "medium"
  },
  {
    "original_label": "Tree Philodendron (Philodendron bipinnatifidum)",
    "vietnamese_name": "Trầu bà thanh xuân",
    "baseline_f1": 59.26,
    "baseline_recall": 53.33,
    "train_30k": 229,
    "priority_level": "medium"
  },
  {
    "original_label": "Crown of Thorns (Euphorbia milii)",
    "vietnamese_name": "Xương rồng bát tiên",
    "baseline_f1": 59.57,
    "baseline_recall": 50.0,
    "train_30k": 216,
    "priority_level": "medium"
  },
  {
    "original_label": "Yew Plum Pine (Podocarpus macrophyllus)",
    "vietnamese_name": "Tùng la hán",
    "baseline_f1": 60.0,
    "baseline_recall": 55.56,
    "train_30k": 207,
    "priority_level": "medium"
  },
  {
    "original_label": "Cast Iron Plant (Aspidistra elatior)",
    "vietnamese_name": "Tỏi rừng",
    "baseline_f1": 60.0,
    "baseline_recall": 72.41,
    "train_30k": 228,
    "priority_level": "watch"
  },
  {
    "original_label": "Arrowhead Plant (Syngonium podophyllum)",
    "vietnamese_name": "Trầu bà chân vịt",
    "baseline_f1": 61.22,
    "baseline_recall": 57.69,
    "train_30k": 199,
    "priority_level": "medium"
  },
  {
    "original_label": "Fan Palm - Lady Palm (Rhapis excelsa)",
    "vietnamese_name": "Trúc mây",
    "baseline_f1": 61.76,
    "baseline_recall": 80.77,
    "train_30k": 196,
    "priority_level": "watch"
  },
  {
    "original_label": "Rubber Plant (Ficus elastica)",
    "vietnamese_name": "Đa búp đỏ",
    "baseline_f1": 62.22,
    "baseline_recall": 93.33,
    "train_30k": 232,
    "priority_level": "watch"
  },
  {
    "original_label": "Giant Taro (Alocasia macrorrhizos)",
    "vietnamese_name": "Ráy khổng lồ",
    "baseline_f1": 63.64,
    "baseline_recall": 56.0,
    "train_30k": 196,
    "priority_level": "medium"
  },
  {
    "original_label": "Coleus (Coleus scutellarioides)",
    "vietnamese_name": "Tía tô cảnh",
    "baseline_f1": 63.83,
    "baseline_recall": 57.69,
    "train_30k": 196,
    "priority_level": "medium"
  },
  {
    "original_label": "Yucca",
    "vietnamese_name": "Cây Yucca",
    "baseline_f1": 66.67,
    "baseline_recall": 66.67,
    "train_30k": 62,
    "priority_level": "high"
  },
  {
    "original_label": "Bird of Paradise (Strelitzia reginae)",
    "vietnamese_name": "Thiên điểu",
    "baseline_f1": 66.67,
    "baseline_recall": 82.61,
    "train_30k": 178,
    "priority_level": "watch"
  },
  {
    "original_label": "Parrot Flower (Heliconia psittacorum)",
    "vietnamese_name": "Chuối mỏ két",
    "baseline_f1": 66.67,
    "baseline_recall": 55.56,
    "train_30k": 208,
    "priority_level": "medium"
  },
  {
    "original_label": "Monstera Deliciosa (Monstera deliciosa)",
    "vietnamese_name": "Trầu bà lá xẻ Monstera",
    "baseline_f1": 67.14,
    "baseline_recall": 85.45,
    "train_30k": 435,
    "priority_level": "watch"
  },
  {
    "original_label": "Coralberry (Ardisia crenata)",
    "vietnamese_name": "Kim ngân lượng",
    "baseline_f1": 68.42,
    "baseline_recall": 56.52,
    "train_30k": 174,
    "priority_level": "medium"
  },
  {
    "original_label": "Raindrop Peperomia (Peperomia polybotrya)",
    "vietnamese_name": "Trường sinh giọt nước",
    "baseline_f1": 70.27,
    "baseline_recall": 68.42,
    "train_30k": 142,
    "priority_level": "high"
  },
  {
    "original_label": "Rose-painted Calathea (Calathea roseopicta)",
    "vietnamese_name": "Đuôi công hồng",
    "baseline_f1": 75.0,
    "baseline_recall": 68.18,
    "train_30k": 164,
    "priority_level": "medium"
  },
  {
    "original_label": "Asparagus Fern (Asparagus setaceus)",
    "vietnamese_name": "Thiên môn đông cảnh",
    "baseline_f1": 78.05,
    "baseline_recall": 76.19,
    "train_30k": 156,
    "priority_level": "medium"
  },
  {
    "original_label": "Black Velvet Alocasia (Alocasia reginula)",
    "vietnamese_name": "Môn nhung đen",
    "baseline_f1": 78.95,
    "baseline_recall": 71.43,
    "train_30k": 158,
    "priority_level": "medium"
  },
  {
    "original_label": "Blushing Bromeliad (Neoregelia carolinae)",
    "vietnamese_name": "Dứa cảnh Neoregelia",
    "baseline_f1": 80.85,
    "baseline_recall": 86.36,
    "train_30k": 176,
    "priority_level": "watch"
  },
  {
    "original_label": "Ponytail Palm (Beaucarnea recurvata)",
    "vietnamese_name": "Cây chân voi",
    "baseline_f1": 89.8,
    "baseline_recall": 95.65,
    "train_30k": 175,
    "priority_level": "watch"
  },
  {
    "original_label": "Round-leaf Calathea (Calathea orbifolia)",
    "vietnamese_name": "Đuôi công lá tròn",
    "baseline_f1": 90.91,
    "baseline_recall": 90.91,
    "train_30k": 171,
    "priority_level": "watch"
  },
  {
    "original_label": "Sago Palm (Cycas revoluta)",
    "vietnamese_name": "Vạn tuế",
    "baseline_f1": 95.24,
    "baseline_recall": 100.0,
    "train_30k": 160,
    "priority_level": "medium"
  }
]
PRIORITY_MAP = {p['original_label']: p for p in PRIORITY_48}

print('=' * 70)
print('🚀 SẴN SÀNG TRAIN TĂNG CƯỜNG 48 LOÀI ƯU TIÊN:')
print('  • TensorFlow :', tf.__version__)
print('  • GPU        :', tf.config.list_physical_devices('GPU'))
print('  • File Zip   :', ZIP_30K_PATH)
print('  • Lưu kết quả:', SAVE_ROOT)
print('  • Số loài ưu tiên:', len(PRIORITY_48))
print('=' * 70)


In [ ]:
# ==============================================================================
# BƯỚC 2: GIẢI NÉN, LỌC TRÙNG MD5 & CHIA STRATIFIED 80% TRAIN - 10% VAL - 10% TEST
# ==============================================================================
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

if not RAW_DIR.exists() or len(list(RAW_DIR.iterdir())) == 0:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    print('📦 Đang giải nén bộ dữ liệu ra SSD Colab...')
    with zipfile.ZipFile(ZIP_30K_PATH, 'r') as zf:
        zf.extractall(RAW_DIR)

# Tìm đúng thư mục chứa 90 folder loài cây
def find_dataset_root(root: Path) -> Path:
    subdirs = [d for d in root.iterdir() if d.is_dir() and not d.name.startswith('__')]
    if len(subdirs) >= 80:
        return root
    for d in subdirs:
        sub_sub = [x for x in d.iterdir() if x.is_dir() and not x.name.startswith('__')]
        if len(sub_sub) >= 80:
            return d
    return root

DATASET_ROOT = find_dataset_root(RAW_DIR)
class_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir() and not d.name.startswith('__')])
print(f'✅ Tìm thấy {len(class_dirs)} thư mục loài cây tại: {DATASET_ROOT}')

def md5_file(p: Path) -> str:
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
for sp in ('train', 'val', 'test'):
    (SPLIT_DIR / sp).mkdir(parents=True, exist_ok=True)

seen_hashes = set()
dup_count = 0
split_records = []

for cls_dir in class_dirs:
    cls_name = cls_dir.name
    unique_imgs = []
    for img_p in sorted(cls_dir.rglob('*')):
        if not img_p.is_file() or img_p.suffix.lower() not in VALID_EXTS:
            continue
        h = md5_file(img_p)
        if h in seen_hashes:
            dup_count += 1
            continue
        seen_hashes.add(h)
        unique_imgs.append(img_p)

    # Chia 80% Train - 10% Val - 10% Test độc lập (KHÔNG augment vào Val/Test)
    train_files, temp_files = train_test_split(unique_imgs, test_size=0.20, random_state=RANDOM_SEED)
    val_files, test_files = train_test_split(temp_files, test_size=0.50, random_state=RANDOM_SEED)

    for sp_name, flist in (('train', train_files), ('val', val_files), ('test', test_files)):
        dst_cls = SPLIT_DIR / sp_name / cls_name
        dst_cls.mkdir(parents=True, exist_ok=True)
        for idx, src in enumerate(flist):
            shutil.copy2(src, dst_cls / f'{idx:04d}_{src.name}')

    split_records.append({
        'original_label': cls_name,
        'is_priority_48': cls_name in PRIORITY_MAP,
        'train_before': len(train_files),
        'val_fixed': len(val_files),
        'test_fixed': len(test_files),
        'total_before': len(unique_imgs),
    })

df_split = pd.DataFrame(split_records)
df_split.to_csv(SAVE_ROOT / 'split_counts_before_aug.csv', index=False, encoding='utf-8-sig')
print(f'✅ Lọc trùng MD5: bỏ {dup_count} ảnh trùng.')
print(f'✅ Phân chia gốc: Train={df_split["train_before"].sum()} | Val={df_split["val_fixed"].sum()} | Test={df_split["test_fixed"].sum()}')


In [ ]:
# ==============================================================================
# BƯỚC 3: TĂNG CƯỜNG ẢNH (XOAY GÓC, GIẢM/TĂNG SÁNG, LẬT, TƯƠNG PHẢN, ZOOM)
#         CHỈ ÁP DỤNG TRÊN TẬP TRAIN CHO 48 LOÀI ƯU TIÊN (< 60% / ÍT ẢNH)
# ==============================================================================
def normalize_rgb(img: Image.Image) -> Image.Image:
    img = ImageOps.exif_transpose(img)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        rgba = img.convert('RGBA')
        bg = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
        return Image.alpha_composite(bg, rgba).convert('RGB')
    return img.convert('RGB')

def augment_plant_image(img: Image.Image, rng: random.Random) -> Image.Image:
    res = normalize_rgb(img)
    # 1. Lật ngang (50%)
    if rng.random() < 0.5:
        res = ImageOps.mirror(res)
    # 2. Xoay góc (-25° đến +25°)
    angle = rng.uniform(-25.0, 25.0)
    res = res.rotate(angle, resample=Image.Resampling.BILINEAR, expand=False, fillcolor=(245, 245, 245))
    # 3. Giảm hoặc tăng độ sáng (0.70x - 1.30x)
    bright_factor = rng.choice([rng.uniform(0.70, 0.88), rng.uniform(1.12, 1.30)])
    res = ImageEnhance.Brightness(res).enhance(bright_factor)
    # 4. Tăng/giảm độ tương phản & màu sắc
    res = ImageEnhance.Contrast(res).enhance(rng.uniform(0.80, 1.25))
    res = ImageEnhance.Color(res).enhance(rng.uniform(0.85, 1.20))
    # 5. Zoom / Crop nhẹ (84% - 96%)
    if rng.random() < 0.65:
        w, h = res.size
        pct = rng.uniform(0.84, 0.96)
        nw, nh = max(32, int(w * pct)), max(32, int(h * pct))
        left = rng.randint(0, max(0, w - nw))
        top = rng.randint(0, max(0, h - nh))
        res = res.crop((left, top, left + nw, top + nh))
    # 6. Làm mờ nhẹ mô phỏng rung camera (15%)
    if rng.random() < 0.15:
        res = res.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.4, 1.0)))
    return res.resize(IMAGE_SIZE, Image.Resampling.LANCZOS)

rng = random.Random(RANDOM_SEED)
aug_summary = []
total_generated = 0

for rec in split_records:
    cls_name = rec['original_label']
    train_cls_dir = SPLIT_DIR / 'train' / cls_name
    orig_files = sorted([p for p in train_cls_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])
    n_orig = len(orig_files)
    n_add = 0

    if rec['is_priority_48'] and n_orig > 0:
        n_needed = max(TARGET_TRAIN_PER_PRIORITY_CLASS - n_orig, MIN_EXTRA_AUG_FOR_PRIORITY)
        for k in range(n_needed):
            src_img = orig_files[k % n_orig]
            try:
                with Image.open(src_img) as im:
                    aug_im = augment_plant_image(im, rng)
                    out_p = train_cls_dir / f'aug48_{k:04d}_{src_img.stem}.jpg'
                    aug_im.save(out_p, format='JPEG', quality=92)
                    n_add += 1
                    total_generated += 1
            except Exception:
                continue

    aug_summary.append({
        **rec,
        'aug_added': n_add,
        'train_after': n_orig + n_add,
    })

df_aug = pd.DataFrame(aug_summary)
df_aug.to_csv(SAVE_ROOT / 'split_counts_after_aug.csv', index=False, encoding='utf-8-sig')
print(f'✅ Đã sinh tổng cộng {total_generated:,} ảnh tăng cường (xoay góc, giảm/tăng sáng, lật, zoom) cho 48 loài ưu tiên!')
print(f'✅ Tổng tập Train sau tăng cường: {df_aug["train_after"].sum():,} ảnh (Val={df_aug["val_fixed"].sum():,}, Test={df_aug["test_fixed"].sum():,} giữ nguyên 100%).')
df_aug[df_aug['is_priority_48']].head(10)


In [ ]:
# ==============================================================================
# BƯỚC 4: HUẤN LUYỆN 2 GIAI ĐOẠN (TRANSFER LEARNING + BALANCED CLASS WEIGHTS)
# ==============================================================================
def make_ds(split_name: str, shuffle: bool):
    return keras.utils.image_dataset_from_directory(
        str(SPLIT_DIR / split_name),
        labels='inferred',
        label_mode='int',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=RANDOM_SEED,
    )

train_ds = make_ds('train', shuffle=True)
val_ds = make_ds('val', shuffle=False)
test_ds = make_ds('test', shuffle=False)
class_names = train_ds.class_names
num_classes = len(class_names)

# Tính Balanced Class Weights trên tập Train sau tăng cường
y_train_all = []
for _, y_batch in train_ds.unbatch().batch(2048):
    y_train_all.extend(y_batch.numpy().tolist())
y_train_all = np.array(y_train_all)
cw_vals = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train_all)
class_weights = {int(i): float(w) for i, w in enumerate(cw_vals)}

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

def build_model(arch: str):
    inputs = keras.Input(shape=(224, 224, 3), name='image_input')
    if arch == 'mobilenet_v2':
        x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)
        base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=x)
        fine_lr, unfreeze_n = 5e-5, 40
    elif arch == 'efficientnet_b0':
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
        fine_lr, unfreeze_n = 3e-5, 40
    else:
        raise ValueError(f'Unsupported arch: {arch}')

    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = keras.Model(inputs, outputs, name=f'{arch}_aug48')
    return model, base, fine_lr, unfreeze_n

trained_models = {}
for arch in MODELS_TO_TRAIN:
    print(f'\n==================== TRAINING {arch.upper()} (AUG 48 SPECIES) ====================')
    t0 = time.time()
    model, base, fine_lr, unfreeze_n = build_model(arch)

    # Giai đoạn 1: Feature Extraction
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=12,
        class_weight=class_weights,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ],
        verbose=1,
    )

    # Giai đoạn 2: Fine-Tuning
    base.trainable = True
    for layer in base.layers[:-unfreeze_n]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    best_path = SAVE_ROOT / f'best_{arch}_aug48.keras'
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=fine_lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        class_weight=class_weights,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(best_path), monitor='val_accuracy', save_best_only=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        ],
        verbose=1,
    )
    elapsed_min = round((time.time() - t0) / 60.0, 2)
    trained_models[arch] = {'path': best_path, 'train_minutes': elapsed_min}
    print(f'✅ Hoàn tất {arch} sau {elapsed_min} phút -> {best_path}')


In [ ]:
# ==============================================================================
# BƯỚC 5: ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & XUẤT FILE EXCEL KẾT QUẢ THẬT 100%
# ==============================================================================
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

summary_rows = []
per_class_results = {}

for arch, info in trained_models.items():
    model = keras.models.load_model(info['path'], compile=False)
    probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    top3_hits = [1 if y_true[i] in np.argsort(probs[i])[-3:] else 0 for i in range(len(y_true))]

    top1 = accuracy_score(y_true, y_pred)
    top3 = float(np.mean(top3_hits))
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    m_p, m_r, m_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    per_class_results[arch] = rep_dict

    # Đếm số loài < 60% F1 sau khi tăng cường
    low_f1_cnt = sum(1 for c in class_names if rep_dict[c]['f1-score'] < 0.60)
    low_r_cnt = sum(1 for c in class_names if rep_dict[c]['recall'] < 0.60)

    summary_rows.append({
        'Mo_Hinh': arch,
        'Thoi_Gian_Train_Phut': info['train_minutes'],
        'So_Anh_Test': len(y_true),
        'Top1_Acc_%': round(top1 * 100, 2),
        'Top3_Acc_%': round(top3 * 100, 2),
        'Weighted_Precision_%': round(w_p * 100, 2),
        'Weighted_Recall_%': round(w_r * 100, 2),
        'Weighted_F1_%': round(w_f1 * 100, 2),
        'Macro_F1_%': round(m_f1 * 100, 2),
        'So_Loai_F1_Duoi_60': low_f1_cnt,
        'So_Loai_Recall_Duoi_60': low_r_cnt,
    })

    pd.DataFrame(rep_dict).T.to_csv(SAVE_ROOT / f'classification_report_{arch}_aug48.csv', encoding='utf-8-sig')

df_summary = pd.DataFrame(summary_rows)

# Bảng so sánh 48 loài ưu tiên Trước (Baseline MNV2) vs Sau (Model mới vừa train)
cmp_48_rows = []
for p in PRIORITY_48:
    lbl = p['original_label']
    row = {
        'Ten_Tieng_Viet': p['vietnamese_name'],
        'Ten_Tieng_Anh': lbl,
        'Baseline_MNV2_F1_%': p['baseline_f1'],
        'Baseline_MNV2_Recall_%': p['baseline_recall'],
    }
    for arch in trained_models:
        if lbl in per_class_results[arch]:
            new_f1 = round(per_class_results[arch][lbl]['f1-score'] * 100, 2)
            new_r = round(per_class_results[arch][lbl]['recall'] * 100, 2)
            row[f'{arch}_Sau_F1_%'] = new_f1
            row[f'{arch}_Delta_F1_%'] = round(new_f1 - p['baseline_f1'], 2)
            row[f'{arch}_Sau_Recall_%'] = new_r
    cmp_48_rows.append(row)
df_cmp_48 = pd.DataFrame(cmp_48_rows)

# Bảng chi tiết 90 loài
all_90_rows = []
for lbl in class_names:
    row = {'Ten_Tieng_Anh': lbl, 'Uu_Tien_48': lbl in PRIORITY_MAP}
    for arch in trained_models:
        d = per_class_results[arch][lbl]
        row[f'{arch}_P_%'] = round(d['precision'] * 100, 2)
        row[f'{arch}_R_%'] = round(d['recall'] * 100, 2)
        row[f'{arch}_F1_%'] = round(d['f1-score'] * 100, 2)
        row['Support_Test'] = int(d['support'])
    all_90_rows.append(row)
df_all_90 = pd.DataFrame(all_90_rows)

# Xuất file Excel tổng hợp thẳng lên Google Drive
excel_out = SAVE_ROOT / 'Ket_Qua_Train_Tang_Cuong_48_Loai.xlsx'
with pd.ExcelWriter(excel_out, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Tong_Quan_Ket_Qua', index=False)
    df_cmp_48.to_excel(writer, sheet_name='So_Sanh_48_Loai_Uu_Tien', index=False)
    df_all_90.to_excel(writer, sheet_name='Chi_Tiet_90_Loai', index=False)
    df_aug.to_excel(writer, sheet_name='Phan_Bo_Train_Truoc_Sau', index=False)

with open(SAVE_ROOT / 'metrics_summary.json', 'w', encoding='utf-8') as mf:
    json.dump(summary_rows, mf, ensure_ascii=False, indent=2)

print('🎉 HOÀN TẤT! Kết quả thật đã được lưu tại:', excel_out)
display(df_summary)
